<a href="https://colab.research.google.com/github/tamaramkrtchyann/https-colab.research.google.com-drive-1m1mW5SUrv32zerXcelsNZzBILyX10i1T/blob/main/MMarketcap_sql_project_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Global Companies Market Capitalization:Project Overview
This project explores the dynamics of global market capitalization, examining how capital is distributed among the world’s leading companies. By processing and normalizing a large-scale financial dataset, this study aims to uncover structural patterns in global markets—such as the concentration of wealth within specific industries, the dominance of certain economies, and the overall stratification of firms by size. Rather than just analyzing raw figures, this research provides a clear perspective on the financial landscape, highlighting the significant disparities and trends that define today's global economy.

In [1]:
!git config --global user.email "toma.mkrtchyan.26@email.com"
!git config --global user.name "tamaramkrtchyan"

In [ ]:
# Libraries
import sqlite3
import pandas as pd
import numpy as np
import re
import os


In [ ]:
from google.colab import files


uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)

print("Number of rows:", len(df_raw))

df_raw.head()


Saving companiesmarketcap (1).csv to companiesmarketcap (1).csv
Number of rows: 7988


,Rank,Company Names,Company Code,Marketcap,Stock Price,Origin Flag,Country
0,1.0,Microsoft,MSFT,$3.033 T,$407.21,🇺🇸,USA
1,2.0,Apple,AAPL,$2.951 T,$190.92,🇺🇸,USA
2,3.0,Saudi Aramco,2222.SR,$2.026 T,$8.34,🇸🇦,S. Arabia
3,4.0,Alphabet (Google),GOOG,$1.909 T,$153.46,🇺🇸,USA
4,5.0,Amazon,AMZN,$1.653 T,$160.05,🇺🇸,USA


The table above shows the first few rows of the dataset. This step helps to check whether the file was loaded correctly and gives a quick understanding of the available columns and data format.

## TRANSFORM: Clean & Prepare Data







In this cell, I performed an ETL process to transform the raw data into a clean, analytical format. Using SQL, I extracted the data, then applied data cleaning and type casting using TRIM, REPLACE, and CAST functions. This allowed me to standardize the market cap column by converting text strings into numeric values. By filtering out invalid records, I ensured data integrity, which resulted in a reliable companies_clean table. This step is the foundation of my project, as it normalizes the dataset, enabling accurate and efficient financial analysis.

In [ ]:
import sqlite3
import pandas as pd
from google.colab import files

# 1. Upload CSV
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 2. Connection
conn = sqlite3.connect('marketcap.db')

# 3. Load raw data
df_raw = pd.read_csv(filename)
df_raw.to_sql('companies_raw', conn, if_exists='replace', index=False)
print(f" Loaded {len(df_raw)} rows into companies_raw")

# 4. Clean table
conn.execute("DROP TABLE IF EXISTS companies_clean")

conn.execute("""
CREATE TABLE companies_clean AS
WITH calculated_data AS (
    SELECT
        TRIM([Company Names]) AS company_name,
        TRIM([Company Code])  AS ticker,
        TRIM(Country)         AS country_name,
        CASE
            WHEN [Marketcap] LIKE '%T' THEN CAST(REPLACE(REPLACE([Marketcap], '$', ''), 'T', '') AS REAL) * 1e12
            WHEN [Marketcap] LIKE '%B' THEN CAST(REPLACE(REPLACE([Marketcap], '$', ''), 'B', '') AS REAL) * 1e9
            WHEN [Marketcap] LIKE '%M' THEN CAST(REPLACE(REPLACE([Marketcap], '$', ''), 'M', '') AS REAL) * 1e6
            ELSE 0
        END AS marketcap_raw
    FROM companies_raw
    WHERE [Marketcap] IS NOT NULL
)
SELECT
    *,
    (marketcap_raw * 100.0 / SUM(marketcap_raw) OVER ()) AS market_share_percent
FROM calculated_data
""")

conn.commit()
print("✅ companies_clean created")

# 5. Preview
df_cleaned = pd.read_sql("SELECT * FROM companies_clean ORDER BY marketcap_raw DESC LIMIT 10", conn)
display(df_cleaned.style.format({
    'marketcap_raw': '{:,.0f}',
    'market_share_percent': '{:.4f}%'
}))

Saving companiesmarketcap (1).csv to companiesmarketcap (1) (1).csv
 Loaded 7988 rows into companies_raw
✅ companies_clean created


,company_name,ticker,country_name,marketcap_raw,market_share_percent
0,Microsoft,MSFT,USA,"3,033,000,000,000",3.0964%
1,Apple,AAPL,USA,"2,951,000,000,000",3.0126%
2,Saudi Aramco,2222.SR,S. Arabia,"2,026,000,000,000",2.0683%
3,Alphabet (Google),GOOG,USA,"1,909,000,000,000",1.9489%
4,Amazon,AMZN,USA,"1,653,000,000,000",1.6875%
5,NVIDIA,NVDA,USA,"1,522,000,000,000",1.5538%
6,Meta Platforms (Facebook),META,USA,"1,027,000,000,000",1.0485%
7,Berkshire Hathaway,BRK-B,USA,"826,220,000,000",0.8435%
8,Eli Lilly,LLY,USA,"611,430,000,000",0.6242%
9,TSMC,TSM,Taiwan,"602,860,000,000",0.6155%


## Database Schema: Relational Structure





To ensure data quality, I organized the dataset into 5 linked tables: Countries, Industries, Categories, Currencies, and Companies. This "relational" structure avoids data repetition and allows for precise filtering. By using unique IDs to connect the tables, I’ve built a robust and professional foundation for all my financial analysis.

In [ ]:
SCHEMA_SQL = """
-- 1. Countries table
CREATE TABLE IF NOT EXISTS countries (
    country_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_name TEXT NOT NULL UNIQUE
);

-- 2. Industries table
CREATE TABLE IF NOT EXISTS industries (
    industry_id INTEGER PRIMARY KEY AUTOINCREMENT,
    industry_name TEXT NOT NULL UNIQUE
);

-- 3. Categories table (Market cap segments)
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
);

-- 4. Currencies table
CREATE TABLE IF NOT EXISTS currencies (
    currency_id INTEGER PRIMARY KEY AUTOINCREMENT,
    currency_code TEXT NOT NULL UNIQUE
);

-- 5. Companies (The main table)
CREATE TABLE IF NOT EXISTS companies (
    company_id INTEGER PRIMARY KEY AUTOINCREMENT,
    company_name TEXT NOT NULL,
    ticker TEXT UNIQUE,
    marketcap_usd REAL,
    stock_price REAL,
    country_id INTEGER REFERENCES countries(country_id),
    industry_id INTEGER REFERENCES industries(industry_id),
    category_id INTEGER REFERENCES categories(category_id),
    currency_id INTEGER REFERENCES currencies(currency_id),
    flag TEXT
);
"""


conn.executescript(SCHEMA_SQL)
conn.commit()

In [ ]:
# Checking tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

              name
0    companies_raw
1  companies_clean
2        countries
3  sqlite_sequence
4       industries
5       categories
6       currencies
7        companies


### 🌍 Query 1 — Top Countries by Total Market Cap (GROUP BY)

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None

sql("""
SELECT
    co.country_name,
    COUNT(c.company_id)                    AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 3) AS total_marketcap_T,
    ROUND(AVG(c.marketcap_usd) / 1e9, 2)  AS avg_marketcap_B,
    ROUND(MAX(c.marketcap_usd) / 1e12, 3) AS top_company_cap_T
FROM companies c
JOIN countries co ON c.country_id = co.country_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY co.country_name
ORDER BY total_marketcap_T DESC
LIMIT 15;
""", "QUERY 1 — Top 15 Countries by Total Market Cap")


QUERY 1 — Top 15 Countries by Total Market Cap


,country_name,num_companies,total_marketcap_T,avg_marketcap_B,top_company_cap_T


###  Query 2 — Companies per Cap Category (Grouped Analysis)

In [ ]:
def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return result

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None

sql("""
SELECT
    ca.category_name,
    COUNT(*)                                      AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)        AS total_cap_T,
    ROUND(100.0 * SUM(c.marketcap_usd)
          / SUM(SUM(c.marketcap_usd)) OVER (), 2) AS share_of_market_pct,
    ROUND(AVG(c.marketcap_usd) / 1e9, 2)         AS avg_cap_B,
    ROUND(100.0 * COUNT(*)
          / SUM(COUNT(*)) OVER (), 2)             AS share_of_companies_pct
FROM companies c
JOIN categories ca ON c.category_id = ca.category_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY ca.category_name
ORDER BY total_cap_T DESC;
""", "QUERY 2 — Cap Category Inequality: Few companies, most capital")


QUERY 2 — Cap Category Inequality: Few companies, most capital


,category_name,num_companies,total_cap_T,share_of_market_pct,avg_cap_B,share_of_companies_pct


###  Query 3 — Top 10 countries by number of companies

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None
sql("""
SELECT
    co.country_name,
    COUNT(c.company_id)                           AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)        AS total_cap_T,
    ROUND(100.0 * COUNT(*)
          / SUM(COUNT(*)) OVER (), 2)             AS share_of_companies_pct,
    ROUND(100.0 * SUM(c.marketcap_usd)
          / SUM(SUM(c.marketcap_usd)) OVER (), 2) AS share_of_market_pct
FROM companies c
JOIN countries co ON c.country_id = co.country_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY co.country_name
ORDER BY num_companies DESC
LIMIT 10;
""", "Top 10 Countries by Number of Companies")


Top 10 Countries by Number of Companies


,country_name,num_companies,total_cap_T,share_of_companies_pct,share_of_market_pct


## Query4 - Which Countries Dominate Each Cap Category

In [ ]:

sql("""
SELECT
    ca.category_name,
    co.country_name,
    COUNT(*)                                        AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)          AS total_cap_T,
    ROUND(100.0 * COUNT(*)
          / SUM(COUNT(*)) OVER (PARTITION BY ca.category_name), 1) AS pct_of_category
FROM companies c
JOIN countries  co ON c.country_id  = co.country_id
JOIN categories ca ON c.category_id = ca.category_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY ca.category_name, co.country_name
HAVING COUNT(*) >= 5
ORDER BY ca.category_name, num_companies DESC;
""", "Which Countries Dominate Each Cap Category?")


Which Countries Dominate Each Cap Category?


,category_name,country_name,num_companies,total_cap_T,pct_of_category
0,Large Cap,USA,576,22.82,39.0
1,Large Cap,Japan,118,3.58,8.0
2,Large Cap,China,116,4.36,7.9
3,Large Cap,India,90,2.61,6.1
4,Large Cap,Canada,55,2.05,3.7
...,...,...,...,...,...
139,Small Cap,Qatar,6,0.01,0.3
140,Small Cap,Turkey,6,0.01,0.3
141,Small Cap,Cambodia,5,0.00,0.2
142,Small Cap,Greece,5,0.01,0.2




```
# This is formatted as code
```

### 🔍 Query 5 — Bottom 10 Companies (with both price & marketcap data)

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None
sql("""
SELECT
    CASE
        WHEN co.country_name IN ('United States', 'Canada', 'Brazil', 'Mexico')
            THEN ' Americas'
        WHEN co.country_name IN ('China', 'Japan', 'India', 'South Korea',
            'Taiwan', 'Hong Kong', 'Singapore', 'Australia')
            THEN ' Asia-Pacific'
        WHEN co.country_name IN ('Germany', 'France', 'Netherlands', 'Sweden',
            'Switzerland', 'Denmark', 'Italy', 'Spain', 'United Kingdom',
            'Norway', 'Finland', 'Belgium', 'Ireland')
            THEN ' Europe'
        ELSE ' Other'
    END AS region,
    COUNT(c.company_id)                             AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)          AS total_cap_T,
    ROUND(100.0 * SUM(c.marketcap_usd)
          / SUM(SUM(c.marketcap_usd)) OVER (), 2)  AS global_share_pct,
    ROUND(AVG(c.marketcap_usd) / 1e9, 2)           AS avg_cap_B,
    ROUND(MAX(c.marketcap_usd) / 1e12, 3)          AS biggest_company_T
FROM companies c
JOIN countries co ON c.country_id = co.country_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY region
ORDER BY total_cap_T DESC;
""", " Americas vs Asia-Pacific vs Europe — Who Dominates?")


🌍 Americas vs Asia-Pacific vs Europe — Who Dominates?


,region,num_companies,total_cap_T,global_share_pct,avg_cap_B,biggest_company_T
0,🌐 Other,4832,61.96,63.25,12.82,3.033
1,🌏 Asia-Pacific,1667,18.43,18.81,11.05,0.603
2,🌍 Europe,978,13.62,13.91,13.93,0.482
3,🌎 Americas,504,3.94,4.02,7.82,0.138


## Query 6 - Small Country, Big Capital — Highest Avg Cap per Company

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None
sql("""
SELECT
    co.country_name,
    COUNT(c.company_id)                             AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)          AS total_cap_T,
    ROUND(SUM(c.marketcap_usd) / COUNT(c.company_id) / 1e9, 2) AS cap_per_company_B,
    ROUND(100.0 * SUM(c.marketcap_usd)
          / SUM(SUM(c.marketcap_usd)) OVER (), 2)  AS global_share_pct
FROM companies c
JOIN countries co ON c.country_id = co.country_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY co.country_name
HAVING COUNT(c.company_id) BETWEEN 5 AND 100
ORDER BY cap_per_company_B DESC
LIMIT 15;
""", " Small Country, Big Capital — Highest Avg Cap per Company")


🏆 Small Country, Big Capital — Highest Avg Cap per Company


,country_name,num_companies,total_cap_T,cap_per_company_B,global_share_pct
0,S. Arabia,35,2.64,75.43,16.54
1,Russia,17,0.41,23.87,2.54
2,Netherlands,55,1.29,23.48,8.09
3,Ireland,46,0.99,21.58,6.22
4,UAE,36,0.73,20.28,4.57
5,Denmark,51,0.84,16.51,5.28
6,Mexico,30,0.44,14.69,2.76
7,Taiwan,98,1.39,14.23,8.74
8,Spain,59,0.72,12.14,4.49
9,Brazil,80,0.87,10.86,5.44


##Query 7 -Country Rankings — Full Leaderboard

In [ ]:
conn = sqlite3.connect('marketcap.db')

def sql(query, title=""):
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    result = pd.read_sql(query, conn)
    display(result)
    return None
sql("""
SELECT
    RANK() OVER (ORDER BY SUM(c.marketcap_usd) DESC) AS rank,
    co.country_name,
    COUNT(c.company_id)                             AS num_companies,
    ROUND(SUM(c.marketcap_usd) / 1e12, 2)          AS total_cap_T,
    ROUND(100.0 * SUM(c.marketcap_usd)
          / SUM(SUM(c.marketcap_usd)) OVER (), 2)  AS global_share_pct,
    ROUND(AVG(c.marketcap_usd) / 1e9, 2)           AS avg_cap_B
FROM companies c
JOIN countries co ON c.country_id = co.country_id
WHERE c.marketcap_usd IS NOT NULL
GROUP BY co.country_name
ORDER BY total_cap_T DESC
LIMIT 20;
""", " Country Rankings — Full Leaderboard")


🌍 Country Rankings — Full Leaderboard


,rank,country_name,num_companies,total_cap_T,global_share_pct,avg_cap_B
0,1,USA,3648,50.59,51.64,13.87
1,2,China,315,5.84,5.96,18.55
2,3,Japan,349,4.63,4.73,13.27
3,4,India,563,3.90,3.98,6.92
4,5,UK,257,3.17,3.24,12.34
5,6,France,118,2.88,2.94,24.44
6,7,S. Arabia,35,2.64,2.70,75.43
7,8,Canada,394,2.63,2.69,6.68
8,9,Germany,219,2.26,2.31,10.33
9,10,Switzerland,106,2.24,2.29,21.16


::::## Export Results to CSV & Download

In [ ]:

top_companies = pd.read_sql_query("""
    SELECT
        RANK() OVER (ORDER BY c.marketcap_usd DESC) AS rank,
        c.company_name,
        c.ticker,
        ROUND(c.marketcap_usd / 1e9, 2) AS marketcap_B,
        ROUND(c.stock_price, 2)          AS stock_price,
        co.country_name,
        cc.category_name
    FROM companies c
    JOIN countries  co ON c.country_id  = co.country_id
    JOIN categories cc ON c.category_id = cc.category_id
    WHERE c.marketcap_usd IS NOT NULL
    ORDER BY c.marketcap_usd DESC
    LIMIT 500
""", conn)

out_file = 'top500_companies_clean.csv'
top_companies.to_csv(out_file, index=False)
print(f" Saved {len(top_companies)} rows to {out_file}")

from google.colab import files
files.download(out_file)
files.download('marketcap.db')
print(" Files ready for download!")

✅ Saved 500 rows to top500_companies_clean.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Files ready for download!


## Conclusion

Using SQL queries with operations such as GROUP BY, JOIN, filtering, ranking, and aggregation functions, along with Python for data processing and visualization, we analyzed the global distribution of company market capitalization. The analysis revealed that a small number of countries dominate the highest capitalization categories, while most countries hold a significantly smaller share of global market value. This shows that economic power is concentrated rather than equally distributed.
